# Phase 4 — one-command free Colab orchestration
Select a free GPU runtime, configure the documented Colab secrets, then run these cells.

In [ ]:
import os, pathlib, subprocess
from google.colab import userdata
required = ['GH_TOKEN', 'HF_TOKEN', 'PHASE4_HF_REPO', 'PHASE4_GITHUB_REPO', 'PHASE4_GIT_NAME', 'PHASE4_GIT_EMAIL']
values = {name: userdata.get(name) for name in required}
missing = [name for name, value in values.items() if not value]
assert not missing, f'Missing required Colab secrets: {missing}'
for name in ('GH_TOKEN', 'HF_TOKEN', 'PHASE4_HF_REPO', 'PHASE4_GIT_NAME', 'PHASE4_GIT_EMAIL'):
    os.environ[name] = values[name]
project = pathlib.Path('/content/sentimental-analysis-modern')
subprocess.run(['gh', 'auth', 'setup-git'], check=True)
if not project.exists():
    subprocess.run(['git', 'clone', f"https://github.com/{values['PHASE4_GITHUB_REPO']}.git", str(project)], check=True)
subprocess.run(['git', 'config', 'user.name', values['PHASE4_GIT_NAME']], cwd=project, check=True)
subprocess.run(['git', 'config', 'user.email', values['PHASE4_GIT_EMAIL']], cwd=project, check=True)
os.chdir(project)

In [ ]:
# Install the locked base stack, leaving NeoBERT's xformers decision to the orchestrator.
requirements = [line.strip() for line in pathlib.Path('requirements-transformer-smoke.txt').read_text().splitlines() if line.strip() and not line.startswith('#') and not line.startswith('xformers==')]
subprocess.run(['python', '-m', 'pip', 'install', '-q', *requirements], check=True)

In [ ]:
# Change 1 to a larger bounded count only when desired.
!python scripts/phase4_colab_orchestrator.py --max-runs 1